# Week 6 — Project 2: GNN for the Spillover Network (Multi-Market: China / US)
**Owner:** Student A — Line A (spillover dynamics) + Line B (DY-GCN / GAT volatility forecasting)

This notebook ingests Week 3 (VAR/DY connectedness) and Week 4 (EPU/TPU policy uncertainty) outputs.
**Line A now runs as a fully automated loop over `MARKETS` (Requirement 1-4 refactor)**, storing every market's spillover results in `market_summaries`, printing a round-up table per market, and rendering a stacked cross-market TCI comparison plot. Line B (HAR / DY-GCN / GAT) still targets one market at a time — controlled by `MODELING_MARKET` — and its math/NN/statistical logic is unchanged from the single-market version.

**Assumption flagged for review:** the planner specifies the volatility panel and policy-index paths exactly,
but does not give an explicit file for the *rolling DY FEVD matrix series* (only "dynamic row-normalized 11x11
matrices"). This notebook assumes Week 3 exported them to
`outputs/{market}/dy_fevd_matrices.npz` (keys: `dates`, `matrices` shape `(T, 11, 11)`).
If your actual Week 3 export uses a different filename, change `DY_MATRIX_PATH` in Cell 2 only.

If any of the three upstream files are missing, `SYNTHETIC_MODE` auto-activates so the notebook still
runs end-to-end for structural testing — flip it off once real Week 3/4 outputs are in place.


Run this in terminal before importing:

```bash
pip install numpy ipykernel
pip install pandas ipykernel
pip install statsmodels ipykernel
pip install arch ipykernel
pip install openpyxl
pip install matplotlib 
pip install seaborn
pip install torch
pip install scikit-learn
```

In [ ]:
# Cell 1 — Imports
import os, json, math, warnings, itertools
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from torch_geometric.nn import GCNConv, GATConv
    from torch_geometric.data import Data
    HAS_PYG = True
except Exception:
    HAS_PYG = False
    print("torch_geometric not found — GCN/GAT cells will fall back to a dense-matmul implementation "
          "(functionally equivalent for a fixed 11-node graph, just not batched via PyG Data objects).")

from scipy import stats
from sklearn.preprocessing import StandardScaler
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

warnings.filterwarnings("ignore")
np.random.seed(0)
torch.manual_seed(0)


In [ ]:
# Cell 2 — Global Configuration (multi-market, loop-driven)
MARKETS = ['china', 'us']  

MARKET_CONFIGS = {
    'china': {
        'epu': {
            'epu_file':      'China_EPU.xlsx',
            'epu_sheet':     [0, 1, 2],
            'epu_col':       'EPU',
            'is_split_date': True,
            'epu_year_col':  'year',
            'epu_month_col': 'month',
        },
        'tpu': {
            'tpu_file':      'China_Mainland_Paper_TPU.xlsx',
            'tpu_sheet':     3,
            'tpu_col':       'TPU',
            'is_split_date': True,
            'tpu_year_col':  'year',
            'tpu_month_col': 'month',
        },
    },
    'us': {
        'epu': {
            'epu_file':      'US_EPU.xlsx',
            'epu_sheet':     0,
            'epu_col':       'News_Based_Policy_Uncert_Index',
            'is_split_date': True,
            'epu_year_col':  'Year',
            'epu_month_col': 'Month',
        },
        'tpu': {
            'tpu_file':      'US_TPU.xlsx',
            'tpu_sheet':     2,
            'tpu_col':       'TPU',
            'is_split_date': False,
            'date_col':      'DATE',
        },
    },
}

GICS_SECTORS = [
    'Energy', 'Materials', 'Industrials',
    'Consumer Discretionary', 'Consumer Staples', 'Health Care',
    'Financials', 'Information Technology', 'Communication Services',
    'Utilities', 'Real Estate',
]
SECTORS = GICS_SECTORS                       # canonical sector order, shared by every market
n = len(SECTORS)
off_diag_mask = ~np.eye(n, dtype=bool)        # reused by every directional-spillover calc below

HORIZON = 10           # ⚠️ EXPERIMENTATION HANDLE: [WHAT: DY FEVD forecast horizon | WHY: informational —
                       # must match whatever horizon Week 3 used per market | HOW: n/a here, just documents intent]

ROLLING_WINDOW = 200   # ⚠️ EXPERIMENTATION HANDLE: [WHAT: rolling window Week 3 used | WHY: same as above]

LOOKBACK_HAR = {"D": 1, "W": 5, "M": 22}      # used later, Line B only — untouched
TRAIN_FRAC, VAL_FRAC = 0.60, 0.20             # used later, Line B only — untouched

# --- Per-market stress calendars (Requirement 2) --------------------------------------------------
STRESS_WINDOWS_BY_MARKET = {
    "china": {
        "GFC_2008":      ("2008-08-01", "2009-06-30"),
        "CN_2015_Crash": ("2015-06-01", "2016-02-29"),
        "TradeWar_2018": ("2018-03-01", "2018-12-31"),
        "COVID_2020":    ("2020-01-15", "2020-06-30"),
    },
    "us": {
        "GFC_2008":   ("2008-08-01", "2009-06-30"),
        "COVID_2020": ("2020-01-15", "2020-06-30"),
    },
}  # ⚠️ EXPERIMENTATION HANDLE: [WHAT: per-market stress events | WHY: add e.g. "COVID_2020" under "us" to
   # study a market-specific episode | HOW: extend the dict, the clip_stress_windows() call in the loop
   # below picks it up automatically — no other code changes needed]

def clip_stress_windows(windows, data_index, market_label):
    """Drop/clip a market's stress windows against its OWN aligned date range, so a shorter or
    differently-dated sample never crashes downstream shock-reconstruction on an empty slice."""
    if len(data_index) == 0:
        return {}
    lo, hi = data_index.min(), data_index.max()
    clipped = {}
    for name, (start, end) in windows.items():
        start_ts, end_ts = pd.Timestamp(start), pd.Timestamp(end)
        if end_ts < lo or start_ts > hi:
            print(f"  [{market_label}] stress-window '{name}' falls outside data range — dropped")
            continue
        clipped[name] = (max(start_ts, lo).strftime("%Y-%m-%d"), min(end_ts, hi).strftime("%Y-%m-%d"))
    return clipped

MODELING_MARKET = MARKETS[0]  # ⚠️ EXPERIMENTATION HANDLE: [WHAT: which market's data Line B (HAR/DY-GCN/GAT)
                              # trains on | WHY: Line A now runs on every market automatically, but the GNN
                              # forecasting section still targets one market at a time | HOW: set to "us" to
                              # retrain Line B on the US panel instead — no other Line B cell needs editing]


In [ ]:
# Cell 3 — Data Loading & Line A Computation Functions (definitions only)
def _load_policy_index(cfg, key, data_dir):
    """Generic EPU/TPU loader driven entirely by MARKET_CONFIGS[market][key]. Handles multi-sheet +
    split year/month (China), single-sheet + split year/month (US EPU), and single combined date
    column (US TPU) — no hardcoded file/sheet/column name lives here."""
    file_path = data_dir / cfg[f"{key}_file"]
    sheet_spec = cfg[f"{key}_sheet"]
    value_col = cfg[f"{key}_col"]
    sheets = sheet_spec if isinstance(sheet_spec, list) else [sheet_spec]
    frames = [pd.read_excel(file_path, sheet_name=s) for s in sheets]

    if cfg["is_split_date"]:
        year_col, month_col = cfg[f"{key}_year_col"], cfg[f"{key}_month_col"]
        dates = pd.to_datetime(dict(year=frames[0][year_col], month=frames[0][month_col], day=1))
        values = np.mean([f[value_col].values for f in frames], axis=0)  # multi-sheet row-wise mean
    else:
        date_col = cfg["date_col"]
        dates = pd.to_datetime(frames[0][date_col])
        values = frames[0][value_col].values

    series = pd.Series(values, index=pd.DatetimeIndex(dates), name=key.upper()).sort_index()
    return series[~series.index.duplicated(keep="last")]


def _make_synthetic(market, n_days=1200, seed=None):
    """Deterministic-per-market synthetic fallback so the loop below is always runnable, even before
    any real per-market Week 3/4 export exists."""
    seed = seed if seed is not None else (abs(hash(market)) % (2**32))
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2016-01-04", periods=n_days)
    base = np.abs(rng.standard_t(df=5, size=(n_days, n))) * 0.01
    shocks = np.cumsum(rng.normal(0, 0.0005, size=(n_days, n)), axis=0)
    vol_panel = pd.DataFrame(base + np.abs(shocks), index=dates, columns=SECTORS)
    policy = pd.DataFrame({
        "EPU": 100 + np.cumsum(rng.normal(0, 3, size=n_days)),
        "TPU": 100 + np.cumsum(rng.normal(0, 4, size=n_days)),
    }, index=dates)
    base_mat = rng.dirichlet(np.ones(n), size=n)
    mats = np.zeros((n_days, n, n))
    for t in range(n_days):
        drift = rng.normal(0, 0.01, size=(n, n))
        m = np.clip(base_mat + drift * (t / n_days), 0, None)
        m = m / m.sum(axis=1, keepdims=True)
        mats[t] = m
        base_mat = m
    return vol_panel, policy, dates, mats


def load_market_data(market):
    """Requirement 3: resolve context-specific paths, load real production data (falling back to the
    synthetic pipeline if any file is missing), and perform the multi-source datetime alignment
    between the GARCH volatility panel, EPU/TPU indexes, and the FEVD matrix collection."""
    cfg = MARKET_CONFIGS[market]
    data_dir = Path("data") / market
    base_dir = Path("outputs") / market
    vol_panel_path = base_dir / "garch_volatility_panel.csv"
    dy_matrix_path = base_dir / "dy_fevd_matrices.npz"

    synthetic_mode = not (vol_panel_path.exists() and dy_matrix_path.exists() and data_dir.exists())

    if synthetic_mode:
        vol_panel, policy_df, dy_dates, dy_matrices = _make_synthetic(market)
    else:
        vol_panel = pd.read_csv(vol_panel_path, index_col=0, parse_dates=True)[SECTORS]

        epu_series = _load_policy_index(cfg["epu"], "epu", data_dir)
        tpu_series = _load_policy_index(cfg["tpu"], "tpu", data_dir)
        policy_df = pd.concat([epu_series, tpu_series], axis=1)
        # EPU/TPU are monthly by construction — upsample to daily via forward-fill before aligning
        policy_df = policy_df.reindex(vol_panel.index.union(policy_df.index)).sort_index().ffill()

        if not dy_matrix_path.exists():
            raise FileNotFoundError(
                f"No DY FEVD matrix file for market='{market}' at {dy_matrix_path}. "
                f"Confirm Week 3/4 exported this market's rolling connectedness matrices."
            )
        dy_npz = np.load(dy_matrix_path, allow_pickle=True)
        dy_dates = pd.to_datetime(dy_npz["dates"])
        dy_matrices = dy_npz["matrices"]

    # --- multi-index datetime alignment across all three sources (market-agnostic) ---
    common_idx = vol_panel.index.intersection(policy_df.index).intersection(pd.Index(dy_dates))
    vol_panel = vol_panel.loc[common_idx].sort_index()
    policy_df = policy_df.loc[common_idx].sort_index()
    date_to_pos = {d: i for i, d in enumerate(dy_dates)}
    dy_matrices = np.stack([dy_matrices[date_to_pos[d]] for d in common_idx])

    return vol_panel, policy_df, dy_matrices, synthetic_mode


def compute_directional_spillovers(dy_matrices, date_index):
    """Step 1 Spillover Network estimation: TO / FROM / NET directional connectedness per sector,
    per day, plus the system-wide rolling Total Connectedness Index (TCI)."""
    to_series = dy_matrices.transpose(0, 2, 1) * off_diag_mask
    to_total = to_series.sum(axis=1)
    from_total = (dy_matrices * off_diag_mask).sum(axis=2)
    net_total = to_total - from_total

    to_df = pd.DataFrame(to_total, index=date_index, columns=SECTORS)
    from_df = pd.DataFrame(from_total, index=date_index, columns=SECTORS)
    net_df = pd.DataFrame(net_total, index=date_index, columns=SECTORS)
    tci_series = pd.Series((dy_matrices * off_diag_mask).sum(axis=(1, 2)) / n, index=date_index, name="TCI")
    return to_df, from_df, net_df, tci_series


def build_net_directional_table(to_df, from_df, net_df):
    return pd.DataFrame({"TO": to_df.mean(), "FROM": from_df.mean(), "NET": net_df.mean()}) \
             .sort_values("NET", ascending=False)


def classify_roles(net_table, threshold=0.0):
    out = net_table.copy()
    out["Role"] = out["NET"].apply(
        lambda v: "Net Transmitter (systemically dangerous)" if v > threshold
        else "Net Receiver (systemically vulnerable)"
    )
    return out.sort_values("NET", ascending=False)


def shock_reconstruction(stress_windows, net_df, tci_series):
    baseline_net, baseline_tci = net_df.mean(), tci_series.mean()
    report = {}
    for name, (start, end) in stress_windows.items():
        mask = (net_df.index >= start) & (net_df.index <= end)
        if mask.sum() == 0:
            report[name] = {"n_obs": 0, "note": "window not covered by current sample"}
            continue
        window_net = net_df.loc[mask].mean()
        delta = (window_net - baseline_net).sort_values(ascending=False)
        report[name] = {
            "n_obs": int(mask.sum()),
            "tci_during": float(tci_series.loc[mask].mean()),
            "tci_baseline": float(baseline_tci),
            "tci_delta": float(tci_series.loc[mask].mean() - baseline_tci),
            "top_new_transmitters": delta.head(3).round(4).to_dict(),
            "top_new_receivers": delta.tail(3).round(4).to_dict(),
        }
    return report


def policy_sensitivity_report(policy_df, tci_series):
    from sklearn.feature_selection import mutual_info_regression
    policy_aligned = policy_df.reindex(tci_series.index).ffill()

    def association(x, y, x_name):
        pearson_r, pearson_p = stats.pearsonr(x, y)
        spearman_r, spearman_p = stats.spearmanr(x, y)
        mi = mutual_info_regression(x.values.reshape(-1, 1), y.values, random_state=0)[0]
        return {"vs": x_name, "pearson_r": round(pearson_r, 4), "pearson_p": round(pearson_p, 4),
                "spearman_r": round(spearman_r, 4), "spearman_p": round(spearman_p, 4),
                "mutual_info": round(float(mi), 4)}

    return pd.DataFrame([
        association(policy_aligned["EPU"], tci_series, "EPU"),
        association(policy_aligned["TPU"], tci_series, "TPU"),
    ])


## Part 1 — Line A: Spillover Dynamics & Analysis

In [ ]:
# Cell 3B — Automated Execution Loop: Step 1 Spillover Network for every market
# `market_summaries` (Requirement 4: structured storage)
####################################################################################################

market_summaries = {}

for market in MARKETS:
    print(f"\n{'=' * 70}\nProcessing market: {market.upper()}\n{'=' * 70}")

    vol_panel_m, policy_df_m, dy_matrices_m, synthetic_mode_m = load_market_data(market)
    to_df_m, from_df_m, net_df_m, tci_series_m = compute_directional_spillovers(dy_matrices_m, vol_panel_m.index)
    stress_windows_m = clip_stress_windows(STRESS_WINDOWS_BY_MARKET.get(market, {}), vol_panel_m.index, market)

    net_table_m = build_net_directional_table(to_df_m, from_df_m, net_df_m)
    role_table_m = classify_roles(net_table_m)
    shock_report_m = shock_reconstruction(stress_windows_m, net_df_m, tci_series_m)
    policy_sensitivity_m = policy_sensitivity_report(policy_df_m, tci_series_m)

    market_summaries[market] = dict(
        vol_panel=vol_panel_m, policy_df=policy_df_m, dy_matrices=dy_matrices_m,
        synthetic_mode=synthetic_mode_m, stress_windows=stress_windows_m,
        to_df=to_df_m, from_df=from_df_m, net_df=net_df_m, tci_series=tci_series_m,
        net_table=net_table_m, role_table=role_table_m,
        shock_report=shock_report_m, policy_sensitivity=policy_sensitivity_m,
    )
    print(f"[{market}] synthetic_mode={synthetic_mode_m} | shape={vol_panel_m.shape} | "
          f"mean TCI={tci_series_m.mean():.4f}")

print(f"\nCompleted Step 1 Spillover Network estimation for: {list(market_summaries.keys())}")


In [ ]:
# Cell 3C — Round-up: print final tables for every market sequentially
for market in MARKETS:
    s = market_summaries[market]
    print(f"\n{'#' * 70}\n### {market.upper()} — Net Directional Spillover Round-Up\n{'#' * 70}")
    print("\n-- Net Directional Spillover Table (TO / FROM / NET, sample average) --")
    print(s["net_table"].round(4))
    print("\n-- Sector Role Classification --")
    print(s["role_table"][["NET", "Role"]].round(4))
    print("\n-- Policy Sensitivity: TCI vs EPU / TPU --")
    print(s["policy_sensitivity"])
    print("\n-- Shock-Reconstruction Summary (active stress windows) --")
    for event, stats_dict in s["shock_report"].items():
        print(f"  {event}: {stats_dict}")


In [ ]:
# Cell 3D — Combined stacked comparison plot: TCI dynamics across markets
fig, axes = plt.subplots(len(MARKETS), 1, figsize=(11, 4 * len(MARKETS)), sharex=False)
axes = np.atleast_1d(axes)

for ax, market in zip(axes, MARKETS):
    s = market_summaries[market]
    ax.plot(s["tci_series"].index, s["tci_series"].values, lw=1.2, color="#1f4e79")
    ax.set_title(f"Rolling Total Connectedness Index — {market.upper()}"
                 + (" (synthetic)" if s["synthetic_mode"] else ""))
    ax.set_ylabel("TCI")
    for name, (start, end) in s["stress_windows"].items():
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), color="red", alpha=0.08)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle("Network Spillover Connectedness Dynamics — Cross-Market Comparison", y=1.02)
plt.tight_layout()
plt.show()


## Part 2 — Line B: Advanced Modeling for Volatility Forecasting (China)

Target: next-day tail-risk metric derived from the conditional volatility panel (system-wide realized
tail volatility). Three parallel models are trained and compared: **HAR**, **DY-GCN**, **GAT**.

**This section now runs as a loop over every market in `MARKETS`** — data splitting, model training, evaluation, and interpretability all execute sequentially per market, with results stored in `market_results[market]`. Plot titles and file exports are labeled dynamically with the active market.


In [ ]:
# Part 2 Hyperparameters (market-agnostic; defined once before the loop)
TAIL_QUANTILE = 0.90  # ⚠️ EXPERIMENTATION HANDLE: [WHAT: quantile defining "tail" cross-section | WHY: 0.90 is a
                      # moderate tail; too high (0.99) starves training data | HOW: try 0.95 for a stricter
                      # extreme-spillover target, or replace with next-day realized volatility (mean, not quantile)]
REGIME_TCI_QUANTILE = 0.75  # ⚠️ EXPERIMENTATION HANDLE: [WHAT: TCI quantile defining "Turbulent" | WHY: 0.75 is a
                            # moderate stress cutoff | HOW: try 0.90 for a stricter "crisis-only" turbulent regime]

HIDDEN_CHANNELS = 16  # ⚠️ EXPERIMENTATION HANDLE: [WHAT: GCN/GAT hidden width | WHY: with only 11 nodes/day,
                      # wide hidden layers overfit fast | HOW: sweep {8, 16, 32} and watch val QLIKE]
GAT_HEADS = 4           # ⚠️ EXPERIMENTATION HANDLE: [WHAT: GAT attention heads | HOW: sweep {1, 2, 4, 8}]
DROPOUT = 0.1          # ⚠️ EXPERIMENTATION HANDLE: [WHAT: dropout rate | HOW: raise to 0.3-0.5 if the train/val
                       # QLIKE gap widens]

LR = 3e-3              # ⚠️ EXPERIMENTATION HANDLE: [WHAT: Adam learning rate | HOW: sweep {3e-4, 1e-3, 3e-3}]
WEIGHT_DECAY = 1e-4     # ⚠️ EXPERIMENTATION HANDLE: [WHAT: L2 regularization | HOW: sweep {0, 1e-5, 1e-4, 1e-3}]
MAX_EPOCHS = 200
PATIENCE = 15           # ⚠️ EXPERIMENTATION HANDLE: [WHAT: early-stopping patience | HOW: raise to 25-30 if val
                       # loss is still trending down]
CLIP_NORM = 1.0         # ⚠️ EXPERIMENTATION HANDLE: [WHAT: gradient clipping max-norm | HOW: try 0.5 for
                       # tighter control on stress days]
SEEDS = [0, 1, 2]       # ⚠️ EXPERIMENTATION HANDLE: [WHAT: multi-seed averaging pool | HOW: expand to
                       # [0,1,2,3,4] for a tighter confidence band]


In [ ]:
# Feature Engineering & Graph-Tensor Helper Functions (market-agnostic)
def har_features(series, lookbacks=LOOKBACK_HAR):
    df = pd.DataFrame(index=series.index)
    df["D"] = series.rolling(lookbacks["D"]).mean()
    df["W"] = series.rolling(lookbacks["W"]).mean()
    df["M"] = series.rolling(lookbacks["M"]).mean()
    return df

edge_index_full = torch.tensor(
    [[i, j] for i in range(n) for j in range(n) if i != j], dtype=torch.long
).t().contiguous()  # fully connected graph, identical shape for every market (11 nodes)

def build_day_tensor(x_row):
    return torch.tensor(x_row.values, dtype=torch.float32).unsqueeze(1)

def edge_weights_from_dy(dy_matrix):
    return torch.tensor(
        [dy_matrix[i, j] for i in range(n) for j in range(n) if i != j], dtype=torch.float32
    )

def make_split_tensors(X_block, y_block, idx_block, panel_index, dy_matrices):
    dy_pos = {d: i for i, d in enumerate(panel_index)}
    xs, ews, ys = [], [], []
    for d in idx_block:
        xs.append(build_day_tensor(X_block.loc[d]))
        ews.append(edge_weights_from_dy(dy_matrices[dy_pos[d]]))
        ys.append(y_block.loc[d])
    return xs, ews, torch.tensor(ys, dtype=torch.float32)

def dense_adj_from_edge_weights(ew):
    adj = torch.eye(n)
    idx = 0
    for i in range(n):
        for j in range(n):
            if i != j:
                adj[i, j] = ew[idx]
                idx += 1
    return adj


In [ ]:
# Model Definitions (DY-GCN / GAT) & Loss Functions — unchanged, market-agnostic
class DenseGraphConv(nn.Module):
    """Dense-matmul fallback GCN layer used only if torch_geometric is unavailable."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        return self.lin(adj @ x)

class DYGCN(nn.Module):
    """Spatial GCN using the fixed/rolling row-normalized econometric DY matrix as a NON-trainable adjacency."""
    def __init__(self, in_dim=1, hidden=HIDDEN_CHANNELS, dropout=DROPOUT):
        super().__init__()
        self.use_pyg = HAS_PYG
        if self.use_pyg:
            self.conv1 = GCNConv(in_dim, hidden)
            self.conv2 = GCNConv(hidden, hidden)
        else:
            self.conv1 = DenseGraphConv(in_dim, hidden)
            self.conv2 = DenseGraphConv(hidden, hidden)
        self.readout = nn.Linear(hidden, 1)
        self.dropout = dropout

    def forward(self, x, edge_index=None, edge_weight=None, adj=None):
        if self.use_pyg:
            h = F.relu(self.conv1(x, edge_index, edge_weight))
            h = F.dropout(h, p=self.dropout, training=self.training)
            h = F.relu(self.conv2(h, edge_index, edge_weight))
        else:
            h = F.relu(self.conv1(x, adj))
            h = F.dropout(h, p=self.dropout, training=self.training)
            h = F.relu(self.conv2(h, adj))
        pooled = h.mean(dim=0)
        return self.readout(pooled)

class SectorGAT(nn.Module):
    """GAT learning a fully data-driven spatial topology end-to-end (starts fully connected)."""
    # def __init__(self, in_dim=1, hidden=HIDDEN_CHANNELS, heads=GAT_HEADS, dropout=DROPOUT):
    #     super().__init__()
    #     self.use_pyg = HAS_PYG
    #     if self.use_pyg:
    #         self.gat1 = GATConv(in_dim, hidden, heads=heads, dropout=dropout)
    #         self.gat2 = GATConv(hidden * heads, hidden, heads=1, concat=False, dropout=dropout)
    #     else:
    #         self.q = nn.Linear(in_dim, hidden)
    #         self.k = nn.Linear(in_dim, hidden)
    #         self.v = nn.Linear(in_dim, hidden)
    #         self.out_lin = nn.Linear(hidden, hidden)
    #     self.readout = nn.Linear(hidden, 1)
    #     self.dropout = dropout
    #     self.last_attention = None

    # def forward(self, x, edge_index=None):
    #     if self.use_pyg:
    #         h, (ei, alpha) = self.gat1(x, edge_index, return_attention_weights=True)
    #         self.last_attention = (ei.detach(), alpha.detach())
    #         h = F.elu(h)
    #         h = F.dropout(h, p=self.dropout, training=self.training)
    #         h = F.elu(self.gat2(h, edge_index))
    #     else:
    #         q, k, v = self.q(x), self.k(x), self.v(x)
    #         scores = (q @ k.t()) / math.sqrt(q.shape[-1])
            
    #         # ——————————————————————————————————————————————————————————————————————————————
    #         # --- DIAGNOSTIC: Raw Logit Inspection (pre-softmax) ---
    #         print("--- DIAGNOSTIC 2: Raw Logit Inspection ---")
    #         print(f"Raw Scores - Min: {scores.min().item():.4f}, Max: {scores.max().item():.4f}")
    #         print(f"Raw Scores Sample Matrix:\n{scores[0] if len(scores.shape)==3 else scores}")
    #         print("------------------------------------------")
    #         # ——————————————————————————————————————————————————————————————————————————————
    #         # --- DIAGNOSTIC: Raw Logit Inspection (pre-softmax) ---
            
    #         # ——————————————————————————————————————————————————————————————————————————————
    #         alpha = F.softmax(scores, dim=-1)
            
    #         # ---- DIAGNOSTIC: Check if alpha sums to 1 across rows (softmax validation) ---
    #         print("--- DIAGNOSTIC 1: Softmax Validation ---")
    #         print(f"Alpha Shape: {alpha.shape}")

    #         # If it's a dense 11x11 matrix, sum across the last dimension (columns)
    #         if len(alpha.shape) == 2 and alpha.shape[0] == 11:
    #             row_sums = alpha.sum(dim=-1)
    #             print(f"Row sums (Should all be 1.0):\n{row_sums}")
    #             print(f"Are all rows summing to 1? {torch.allclose(row_sums, torch.ones_like(row_sums))}")
    #         elif len(alpha.shape) == 3: # If batched (Batch, 11, 11)
    #             row_sums = alpha.sum(dim=-1)
    #             print(f"Batch Row sums sample (Should be 1.0):\n{row_sums[0]}")
    #         print("----------------------------------------")
    #         # ——————————————————————————————————————————————————————————————————————————————
    #         # ---- DIAGNOSTIC: Check if alpha sums to 1 across rows (softmax validation) ---

    #         self.last_attention = alpha.detach()
    #         h = F.elu(self.out_lin(alpha @ v))
    #     pooled = h.mean(dim=0)
    #     return self.readout(pooled)
    
    def __init__(self, in_dim=1, hidden=HIDDEN_CHANNELS, heads=GAT_HEADS, dropout=DROPOUT):
        super().__init__()
        self.use_pyg = HAS_PYG
        if self.use_pyg:
            self.gat1 = GATConv(in_dim, hidden, heads=heads, dropout=dropout)
            self.gat2 = GATConv(hidden * heads, hidden, heads=1, concat=False, dropout=dropout)
        else:
            self.q = nn.Linear(in_dim, hidden)
            self.k = nn.Linear(in_dim, hidden)
            self.v = nn.Linear(in_dim, hidden)
            self.out_lin = nn.Linear(hidden, hidden)
            
            # --- FIX: Add LeakyReLU for GAT attention stability ---
            self.leakyrelu = nn.LeakyReLU(negative_slope=0.2)
            
        self.readout = nn.Linear(hidden, 1)
        self.dropout = dropout
        self.last_attention = None
    
    def forward(self, x, edge_index=None):
        if self.use_pyg:
            h, (ei, alpha) = self.gat1(x, edge_index, return_attention_weights=True)
            self.last_attention = (ei.detach(), alpha.detach())
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            h = F.elu(self.gat2(h, edge_index))
        else:
            # 1. Feature Projections
            q, k, v = self.q(x), self.k(x), self.v(x)
            
            # 2. Compute Raw Scores and Scale
            scores = (q @ k.t()) / math.sqrt(q.shape[-1])
            
            # --- FIX: Standard GAT Non-Linearity to prevent microscopic variance ---
            scores = self.leakyrelu(scores)
            
            # ——————————————————————————————————————————————————————————————————————————————
            # # --- DIAGNOSTIC: Raw Logit Inspection (pre-softmax) ---
            # print("--- DIAGNOSTIC 2: Raw Logit Inspection ---")
            # print(f"Raw Scores - Min: {scores.min().item():.4f}, Max: {scores.max().item():.4f}")
            # print(f"Raw Scores Sample Matrix:\n{scores[0] if len(scores.shape)==3 else scores}")
            # print("------------------------------------------")
            # ——————————————————————————————————————————————————————————————————————————————
            
            # 3. Softmax along rows
            alpha = F.softmax(scores, dim=-1)
            
            # ——————————————————————————————————————————————————————————————————————————————
            # ---- DIAGNOSTIC: Check if alpha sums to 1 across rows (softmax validation) ---
            # print("--- DIAGNOSTIC 1: Softmax Validation ---")
            # print(f"Alpha Shape: {alpha.shape}")

            # if len(alpha.shape) == 2 and alpha.shape[0] == 11:
            #     row_sums = alpha.sum(dim=-1)
            #     print(f"Row sums (Should all be 1.0):\n{row_sums}")
            #     print(f"Are all rows summing to 1? {torch.allclose(row_sums, torch.ones_like(row_sums))}")
            # elif len(alpha.shape) == 3: 
            #     row_sums = alpha.sum(dim=-1)
            #     print(f"Batch Row sums sample (Should be 1.0):\n{row_sums[0]}")
            # print("----------------------------------------")
            # ——————————————————————————————————————————————————————————————————————————————

            self.last_attention = alpha.detach()
            h = F.elu(self.out_lin(alpha @ v))
            
        pooled = h.mean(dim=0)
        return self.readout(pooled)

def qlike_loss(y_pred, y_true, eps=1e-8):
    """Asymmetric QLIKE loss — standard primary metric for volatility forecast evaluation."""
    ratio = y_true / (y_pred.clamp(min=eps))
    return (ratio - torch.log(ratio.clamp(min=eps)) - 1).mean()

def pinball_loss(y_pred, y_true, quantile=0.9):
    diff = y_true - y_pred
    return torch.maximum(quantile * diff, (quantile - 1) * diff).mean()


In [ ]:
# Training / Evaluation Helper Functions — unchanged logic, explicitly parameterized
# they behave identically no matter which market's tensors are passed in from the loop below.
####################################################################################################

def run_epoch(model, xs, ews, ys, edge_index, optimizer=None, adj_list=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    for i in range(len(xs)):
        x = xs[i]
        if isinstance(model, DYGCN):
            pred = model(x, edge_index=edge_index, edge_weight=ews[i]) if model.use_pyg \
                else model(x, adj=adj_list[i])
        else:
            pred = model(x, edge_index=edge_index)
        loss = qlike_loss(pred.clamp(min=1e-6), ys[i].clamp(min=1e-6))
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            optimizer.step()
        total_loss += loss.item()
    return total_loss / len(xs)

def train_with_early_stopping(model_cls, seed, train_x, train_ew, train_y, train_adj,
                               val_x, val_ew, val_y, val_adj, **model_kwargs):
    torch.manual_seed(seed)
    model = model_cls(**model_kwargs)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_val, best_state, wait = float("inf"), None, 0
    history = {"train": [], "val": []}
    for epoch in range(MAX_EPOCHS):
        tr_loss = run_epoch(model, train_x, train_ew, train_y, edge_index_full, optimizer, train_adj)
        with torch.no_grad():
            val_loss = run_epoch(model, val_x, val_ew, val_y, edge_index_full, None, val_adj)
        history["train"].append(tr_loss)
        history["val"].append(val_loss)
        if val_loss < best_val - 1e-5:
            best_val, best_state, wait = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, history, best_val

def predict_gnn(model, xs, ews, adj_list):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(len(xs)):
            if isinstance(model, DYGCN):
                p = model(xs[i], edge_index=edge_index_full, edge_weight=ews[i]) if model.use_pyg \
                    else model(xs[i], adj=adj_list[i])
            else:
                p = model(xs[i], edge_index=edge_index_full)
            preds.append(p.item())
    return np.array(preds)

def seed_avg_predictions(model_list, xs, ews, adj_list):
    all_preds = np.stack([predict_gnn(m, xs, ews, adj_list) for m in model_list])
    return all_preds.mean(axis=0)

def summarize(y_true, y_pred, label):
    t = torch.tensor(y_true, dtype=torch.float32)
    p = torch.tensor(np.clip(y_pred, 1e-6, None), dtype=torch.float32)
    return {"model": label,
            "QLIKE": round(qlike_loss(p, t).item(), 5),
            "Pinball@0.9": round(pinball_loss(p, t, quantile=TAIL_QUANTILE).item(), 5)}

def diebold_mariano(y_true, pred_a, pred_b, loss="qlike"):
    """DM test comparing forecast loss differential between model A and model B.
    Positive DM stat => model A has higher loss (worse) than model B."""
    y_true = np.asarray(y_true); pred_a = np.asarray(pred_a); pred_b = np.asarray(pred_b)
    eps = 1e-6
    if loss == "qlike":
        la = y_true / np.clip(pred_a, eps, None) - np.log(y_true / np.clip(pred_a, eps, None)) - 1
        lb = y_true / np.clip(pred_b, eps, None) - np.log(y_true / np.clip(pred_b, eps, None)) - 1
    else:
        la = (y_true - pred_a) ** 2
        lb = (y_true - pred_b) ** 2
    d = la - lb
    T = len(d)
    d_bar = d.mean()
    var_d = np.var(d, ddof=0) / T
    dm_stat = d_bar / np.sqrt(var_d + eps)
    p_value = 2 * (1 - stats.norm.cdf(np.abs(dm_stat)))
    return {"DM_stat": round(float(dm_stat), 4), "p_value": round(float(p_value), 5),
            "mean_loss_diff (A - B)": round(float(d_bar), 6)}

def regime_summary(mask, label_true, test_y_np, preds_by_model):
    """preds_by_model: dict like {'HAR': har_pred_np, 'DY-GCN': pred_gcn, 'GAT': pred_gat}"""
    label = "Turbulent" if label_true else "Calm"
    idx_mask = mask.values if label_true else ~mask.values
    if idx_mask.sum() == 0:
        return []
    rows = []
    for name, preds in preds_by_model.items():
        s = summarize(test_y_np[idx_mask], preds[idx_mask], name)
        s["regime"] = label
        s["n_obs"] = int(idx_mask.sum())
        rows.append(s)
    return rows

def average_gat_attention(model, xs, edge_index_full):
    model.eval()
    acc = torch.zeros(n, n)
    with torch.no_grad():
        for i in range(len(xs)):
            _ = model(xs[i], edge_index=edge_index_full)
            if model.use_pyg:
                ei, alpha = model.last_attention
                alpha = alpha.mean(dim=-1) if alpha.dim() > 1 else alpha
                for k in range(ei.shape[1]):
                    acc[ei[0, k], ei[1, k]] += alpha[k].item()
            else:
                acc += model.last_attention
    return (acc / len(xs)).numpy()


### Part 2 Execution Loop

Runs the full HAR / DY-GCN / GAT pipeline — split, train, evaluate, DM test, regime split, interpretability — for every market in `MARKETS`, storing everything in `market_results`.


In [ ]:
# PART 2 EXECUTION LOOP — data splitting, model training, and performance logging,
# now running sequentially for every market in MARKETS.
# now running sequentially for every market in MARKETS. Results stored in `market_results`.
####################################################################################################

market_results = {}

for market in MARKETS:
    print("\n" + "=" * 80)
    print(f" STARTING VOLATILITY FORECASTING PIPELINE: {market.upper()} MARKET")
    print("=" * 80)

    # ------------------------------------------------------------------------
    # 2.1 DATA EXTRACTION FROM PART 1 STORAGE
    # ------------------------------------------------------------------------
    if market not in market_summaries:
        raise KeyError(f"Market '{market}' data not found in market_summaries. Run Part 1 first.")

    vol_panel = market_summaries[market]["vol_panel"]
    dy_matrices = market_summaries[market]["dy_matrices"]
    tci_series = market_summaries[market]["tci_series"]

    print(f"[{market.upper()}] Successfully loaded vol_panel with shape: {vol_panel.shape}")
    print(f"[{market.upper()}] Successfully loaded {len(dy_matrices)} daily adjacency matrices.")

    # ------------------------------------------------------------------------
    # 2.2 CHRONOLOGICAL DATA SPLITTING & FEATURE PREPARATION
    # ------------------------------------------------------------------------
    system_vol = vol_panel.mean(axis=1)
    tail_target = vol_panel.quantile(TAIL_QUANTILE, axis=1).shift(-1)
    tail_target.name = "target_tail_vol"

    panel = vol_panel.join(tail_target).dropna()
    T = len(panel)
    train_end = int(T * TRAIN_FRAC)
    val_end = int(T * (TRAIN_FRAC + VAL_FRAC))
    idx_train = panel.index[:train_end]
    idx_val = panel.index[train_end:val_end]
    idx_test = panel.index[val_end:]
    print(f"[{market.upper()}] train={len(idx_train)}  val={len(idx_val)}  test={len(idx_test)} "
          f"| range {panel.index[0].date()} -> {panel.index[-1].date()}")

    scaler = StandardScaler().fit(panel.loc[idx_train, SECTORS].values)

    def scale_block(idx, _panel=panel, _scaler=scaler):
        return pd.DataFrame(_scaler.transform(_panel.loc[idx, SECTORS].values), index=idx, columns=SECTORS)

    X_train, X_val, X_test = scale_block(idx_train), scale_block(idx_val), scale_block(idx_test)
    y_train = panel.loc[idx_train, "target_tail_vol"]
    y_val = panel.loc[idx_val, "target_tail_vol"]
    y_test = panel.loc[idx_test, "target_tail_vol"]

    # --- HAR benchmark -----------------------------------------------------
    har_X = har_features(system_vol).shift(1)
    har_full = har_X.join(tail_target).dropna()
    har_train = har_full.loc[har_full.index.isin(idx_train)]
    har_test = har_full.loc[har_full.index.isin(idx_test)]
    har_model = OLS(har_train["target_tail_vol"], add_constant(har_train[["D", "W", "M"]])).fit()
    har_pred_test = har_model.predict(add_constant(har_test[["D", "W", "M"]], has_constant="add"))
    har_pred_np = har_pred_test.reindex(idx_test).values

    # --- Graph tensors for DY-GCN / GAT ------------------------------------
    train_x, train_ew, train_y_t = make_split_tensors(X_train, y_train, idx_train, panel.index, dy_matrices)
    val_x, val_ew, val_y_t = make_split_tensors(X_val, y_val, idx_val, panel.index, dy_matrices)
    test_x, test_ew, test_y_t = make_split_tensors(X_test, y_test, idx_test, panel.index, dy_matrices)

    train_adj = [dense_adj_from_edge_weights(ew) for ew in train_ew] if not HAS_PYG else None
    val_adj = [dense_adj_from_edge_weights(ew) for ew in val_ew] if not HAS_PYG else None
    test_adj = [dense_adj_from_edge_weights(ew) for ew in test_ew] if not HAS_PYG else None

    # --- Train DY-GCN and GAT across multiple seeds -------------------------
    seed_models = {"DY-GCN": [], "GAT": []}
    histories = {"DY-GCN": [], "GAT": []}
    for seed in SEEDS:
        m, h, _ = train_with_early_stopping(DYGCN, seed, train_x, train_ew, train_y_t, train_adj,
                                             val_x, val_ew, val_y_t, val_adj)
        seed_models["DY-GCN"].append(m); histories["DY-GCN"].append(h)
        m, h, _ = train_with_early_stopping(SectorGAT, seed, train_x, train_ew, train_y_t, train_adj,
                                             val_x, val_ew, val_y_t, val_adj)
        seed_models["GAT"].append(m); histories["GAT"].append(h)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, (name, hlist) in zip(axes, histories.items()):
        for i, h in enumerate(hlist):
            ax.plot(h["train"], alpha=0.4, color="tab:blue", label="train" if i == 0 else None)
            ax.plot(h["val"], alpha=0.7, color="tab:orange", label="val" if i == 0 else None)
        ax.set_title(f"[{market.upper()}] {name} loss curves (all seeds)")
        ax.set_xlabel("epoch"); ax.set_ylabel("QLIKE")
        ax.legend()
    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------------------
    # 2.3 EVALUATION & VISUALIZATION
    # ------------------------------------------------------------------------
    test_y_np = test_y_t.numpy()
    pred_gcn = seed_avg_predictions(seed_models["DY-GCN"], test_x, test_ew, test_adj)
    pred_gat = seed_avg_predictions(seed_models["GAT"], test_x, test_ew, test_adj)

    metrics_table = pd.DataFrame([
        summarize(test_y_np, har_pred_np, "HAR"),
        summarize(test_y_np, pred_gcn, "DY-GCN"),
        summarize(test_y_np, pred_gat, "GAT"),
    ])
    print(f"\n[{market.upper()}] Out-of-sample metrics:")
    print(metrics_table)

    plt.figure(figsize=(14, 7))
    plt.plot(idx_test, test_y_np, label="Actual Realized Volatility", color="black", alpha=0.6)
    plt.plot(idx_test, har_pred_np, label="HAR Benchmark", linestyle="--")
    plt.plot(idx_test, pred_gcn, label="DY-GCN (Structural Edges)", alpha=0.8)
    plt.plot(idx_test, pred_gat, label="GAT (Dynamic Multi-Head Attention)", alpha=0.8)
    plt.title(f"[{market.upper()}] Out-of-Sample Volatility Forecasting Performance", fontsize=14, fontweight="bold")
    plt.xlabel("Trading Timeline (Test Horizon)")
    plt.ylabel("Volatility Matrix Score (QLIKE Targeted)")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.show()

    # --- Diebold-Mariano significance test -----------------------------------
    dm_results = pd.DataFrame([
        {"comparison": "GAT vs HAR", **diebold_mariano(test_y_np, pred_gat, har_pred_np)},
        {"comparison": "GAT vs DY-GCN", **diebold_mariano(test_y_np, pred_gat, pred_gcn)},
        {"comparison": "DY-GCN vs HAR", **diebold_mariano(test_y_np, pred_gcn, har_pred_np)},
    ])
    print(f"\n[{market.upper()}] Diebold-Mariano test results:")
    print(dm_results)

    # --- Calm vs Turbulent regime split ---------------------------------------
    tci_test = tci_series.reindex(idx_test)
    turbulent_cut = tci_test.quantile(REGIME_TCI_QUANTILE)
    regime_mask = tci_test > turbulent_cut
    preds_by_model = {"HAR": har_pred_np, "DY-GCN": pred_gcn, "GAT": pred_gat}
    regime_rows = (regime_summary(regime_mask, True, test_y_np, preds_by_model)
                   + regime_summary(regime_mask, False, test_y_np, preds_by_model))
    regime_table = pd.DataFrame(regime_rows)[["regime", "model", "n_obs", "QLIKE", "Pinball@0.9"]] \
        if regime_rows else pd.DataFrame(columns=["regime", "model", "n_obs", "QLIKE", "Pinball@0.9"])
    print(f"\n[{market.upper()}] Calm vs Turbulent regime split:")
    print(regime_table)

    # ------------------------------------------------------------------------
    # 2.4 INTERPRETABILITY — LEARNED GAT ATTENTION VS STATIC DY BASELINE
    # ------------------------------------------------------------------------
    print(f"[{market.upper()}] Generating Graph Interpretability Visualizations...")

    gat_attention_matrix = average_gat_attention(seed_models["GAT"][0], test_x, edge_index_full)
    # Ensure the matrix shape maps strictly to the 11x11 sector nodes — if a multi-head raw tensor
    # ever leaks through unaveraged, collapse the head dimension without touching the spatial dims.
    if gat_attention_matrix.ndim > 2:
        gat_attention_matrix = gat_attention_matrix.mean(axis=0)

    static_baseline_matrix = dy_matrices[[panel.index.get_loc(d) for d in idx_test]].mean(axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Plot 1: Static / Averaged Diebold-Yilmaz Spillover Baseline
    sns.heatmap(
        static_baseline_matrix,
        ax=axes[0],
        cmap="viridis",
        vmin=static_baseline_matrix.min(),
        vmax=static_baseline_matrix.max(),
        xticklabels=SECTORS,
        yticklabels=SECTORS,
        cbar_kws={"label": "Spillover Intensity"},
    )
    axes[0].set_title(f"[{market.upper()}] Baseline Structural Edges (DY Network)", fontsize=12)

    # Plot 2: Learned GAT Attention Weights — explicit per-matrix vmin/vmax fixes the washed-out,
    # monochromatic look that shows up when both panels are (implicitly) normalized to the same scale.
    
    # ----------------------------------------------------------------------
    # print("--- DIAGNOSTIC 3: Visualizer Override Active ---")
    # # Forces a perfect 11x11 uniform matrix of 0.0909
    # # import torch
    # attention_matrix = torch.ones((11, 11)) / 11.0 
    # ----------------------------------------------------------------------
    
    sns.heatmap(
        gat_attention_matrix,
        ax=axes[1],
        cmap="magma",
        vmin=gat_attention_matrix.min(),
        vmax=gat_attention_matrix.max(),
        xticklabels=SECTORS,
        yticklabels=SECTORS,
        cbar_kws={"label": "Attention Weight (Spatial Risk)"},
    )
    axes[1].set_title(f"[{market.upper()}] Learned Dynamic GAT Attention Weights", fontsize=12)

    plt.tight_layout()
    plt.show()

    attention_export_path = f"gat_vs_dy_attention_export_{market}.json"
    with open(attention_export_path, "w") as f:
        json.dump({"dy_baseline": static_baseline_matrix.tolist(),
                   "gat_attention": gat_attention_matrix.tolist(),
                   "sectors": SECTORS}, f, indent=2)
    print(f"[{market.upper()}] Exported: {attention_export_path}")

    # --- Structured storage (Requirement: central data structure per market) ---
    market_results[market] = dict(
        idx_train=idx_train, idx_val=idx_val, idx_test=idx_test,
        har_model=har_model, metrics_table=metrics_table, dm_results=dm_results,
        regime_table=regime_table, gat_attention=gat_attention_matrix,
        dy_baseline=static_baseline_matrix,
        predictions={"HAR": har_pred_np, "DY-GCN": pred_gcn, "GAT": pred_gat, "actual": test_y_np},
    )

    print(f"[{market.upper()}] Pipeline Execution Complete.\n" + "=" * 80)


### Cross-Market Model Comparison Round-Up


In [ ]:
# Cross-Market Model Comparison Round-Up
for market in MARKETS:
    print(f"\n### {market.upper()} — Out-of-Sample Forecast Metrics ###")
    print(market_results[market]["metrics_table"])
    print(f"\n### {market.upper()} — Diebold-Mariano Significance ###")
    print(market_results[market]["dm_results"])


## Summary

- **Line A** delivers the net directional spillover table, rolling TCI trajectory, stress-window shock
  reconstruction, sector role classification, and EPU/TPU policy-sensitivity diagnostics.
- **Line B** delivers a chronologically clean HAR / DY-GCN / GAT comparison with QLIKE + pinball loss,
  a Diebold-Mariano significance test, a Calm-vs-Turbulent regime split, and a GAT-vs-DY attention
  interpretability export — now looped automatically across every market in `MARKETS`, with results
  stored per market in `market_results`.
- All tunable knobs are marked `⚠️ EXPERIMENTATION HANDLE` inline, so there is no need to hunt through the notebook
  to find what to sweep first (start with `HIDDEN_CHANNELS`, `LR`, and `TAIL_QUANTILE`).
